# Dataset、DataLoader 与数据边界

## 学习目标

能够构建 Dataset、划分训练/验证/测试索引，并解释增强、shuffle 和 batch 的边界。


## 概念模型与执行路径

Dataset 定义如何取得单个样本，DataLoader 定义如何采样、组批和并行加载。训练和验证可以来自同一原始数据，但必须使用独立 transform；测试集不能参与模型选择。


### 实验 1：构造张量数据集并做可复现划分

**实验目的**：从内存中的特征和标签构造一个 map-style Dataset，再把样本索引划分为互不重叠的训练集和验证集。

`features` 由 0 到 39 排成 `(20, 2)`，表示 20 个样本、每个样本 2 个特征。`labels` 根据每行特征和是否大于 30 生成二分类标签，并转为分类任务常用的 `torch.long`。`TensorDataset` 不复制这些张量，而是让相同索引位置的特征和标签组成一个样本；因此 `dataset[i]` 返回 `(features[i], labels[i])`。

`random_split(dataset, [16, 4], ...)` 返回两个 `Subset`。它们共享原始 `dataset`，各自只保存一组索引。固定种子的独立 `Generator` 让划分在重新运行时可复现，又不会重置或污染全局随机数状态。打印结果应为 `16 4 True`：总样本量保持 20，且两组索引没有交集。

**边界说明**：索引不重叠只能避免样本直接重复，不能自动防止同一用户、同一时间窗口或近重复内容跨集合泄漏。真实项目应先按业务实体或时间划分，再交给 Dataset。

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split

features = torch.arange(40, dtype=torch.float32).reshape(20, 2)
labels = (features.sum(dim=1) > 30).long()
dataset = TensorDataset(features, labels)
train_set, validation_set = random_split(dataset, [16, 4], generator=torch.Generator().manual_seed(42))
print(len(train_set), len(validation_set), set(train_set.indices).isdisjoint(validation_set.indices))


### 实验 2：随机采样并组成 mini-batch

**实验目的**：观察 DataLoader 如何把 Dataset 的单个样本变成适合模型输入的批次。`batch_size=6` 时，16 个训练样本会形成 3 个 batch，大小依次为 6、6、4；默认 `drop_last=False`，所以最后不足 6 个的 batch 不会被丢弃。

默认 `collate_fn` 会把 6 个形状为 `(2,)` 的特征沿新 batch 维堆叠成 `(6, 2)`，把 6 个标量标签堆叠成 `(6,)`。最后一个 batch 对应 `(4, 2)` 和 `(4,)`。这说明训练代码不能假设每个 batch 都等于配置的 `batch_size`；若 BatchNorm 或固定形状逻辑确实要求完整批次，可显式设置 `drop_last=True`，但会舍弃部分训练样本。

`shuffle=True` 通常通过随机采样器为每次迭代生成新的索引顺序，不会修改 Dataset 本身。它能减少样本原始排列带来的梯度偏差，但本实验没有给 DataLoader 传 generator，所以每次运行的样本顺序可能不同，形状和样本覆盖范围不变。

**观察重点**：`len(loader)` 是 batch 数 3，而 `len(loader.dataset)` 是样本数 16。二者不要混用。

In [ ]:
loader = DataLoader(train_set, batch_size=6, shuffle=True)
for batch_index, (inputs, targets) in enumerate(loader):
    print(batch_index, inputs.shape, targets.shape)


### 实验 3：定位课程代码的导入根目录

**实验目的**：让 notebook 无论从仓库根目录、notebooks 目录附近还是 `07-deep-learning/pytorch` 目录启动，都能导入课程的 `common` 包。

代码依次检查三个候选路径，并选择第一个包含 `common` 目录的路径作为 `PYTORCH_ROOT`。如果该路径还不在 `sys.path`，就插入到最前面，使下一项实验中的 `from common.data import image_loaders` 能由 Python 导入系统解析。先判断再插入可以避免重复运行单元格后产生多个相同路径。

这段代码解决的是 notebook 工作目录不稳定的问题，不是 Python 包发布方案。`Path.cwd()` 取决于 Jupyter 服务器从哪里启动，而不一定等于 notebook 文件所在目录。若三个候选路径都没有 `common`，`next(...)` 会抛出 `StopIteration`，应先确认工作目录或项目结构。

**工程提示**：正式项目更适合把包以 editable mode 安装到独立环境，或从固定项目入口启动 Jupyter，避免在业务代码中动态修改 `sys.path`。

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 4：加载真实 MNIST 并隔离训练、验证和测试数据流

**实验目的**：使用课程实现的 `common.data.image_loaders` 建立真实图像任务的数据边界，并检查一个 batch 的 NCHW 形状与三个集合的规模。首次运行可能从网络下载 MNIST；已有缓存时会直接复用。

本次参数的含义如下：

- `batch_size=64`：训练 loader 每次最多返回 64 个样本；
- `quick=True`：从 MNIST 训练部分确定性抽取 1024 个索引，其中 20%（204 个）作为验证集、其余 820 个作为训练集，并把测试集限制为 256 个样本；
- `augment=True`：训练样本在 `ToTensor` 和标准化前随机执行小角度、少量平移的 `RandomAffine`；验证集和测试集只做确定性的张量转换与标准化；
- 返回的 `channels=1` 表明 MNIST 是单通道灰度图。

因此首个训练 batch 的 `images` 形状应为 `(64, 1, 28, 28)`，`labels` 为 `(64,)`，集合大小应为 `820 204 256`。训练 loader 使用 shuffle，验证和测试 loader 不打乱；三个 loader 当前都设置 `num_workers=0`，数据读取发生在主进程，适合 notebook 和小规模 quick 实验。

**关键实现**：训练和验证使用相同的一组原始 MNIST 训练数据与互斥索引，但分别创建带训练 transform、评估 transform 的两个 Dataset 实例，再分别包装成 `Subset`。这样既保持划分一致，又不会让随机增强污染验证指标。测试集来自 MNIST 官方 test split，只用于最终评估，不应参与超参数选择。

In [ ]:
# 该单元首次运行会下载真实 MNIST。
from common.data import image_loaders
train_loader, validation_loader, test_loader, channels = image_loaders(
    "mnist", PYTORCH_ROOT / "data", batch_size=64, quick=True, augment=True
)
images, labels = next(iter(train_loader))
print("MNIST batch:", images.shape, labels.shape, "channels:", channels)
print("split sizes:", len(train_loader.dataset), len(validation_loader.dataset), len(test_loader.dataset))


## 底层机制

一次迭代的数据路径可以概括为：Sampler 产生索引 → Dataset 的 `__getitem__` 读取并转换单个样本 → `collate_fn` 合并样本 → DataLoader 把 batch 交给训练循环。Dataset 决定“一个样本是什么”，Sampler 决定“以什么顺序取哪些样本”，batch sampler 决定“哪些索引组成一批”，`collate_fn` 决定“如何合并这一批”。

随机增强通常发生在 Dataset transform 中，并在每次 `__getitem__` 时重新采样随机参数。因此同一训练索引跨 epoch 可能得到不同图像。如果训练和验证 `Subset` 共享同一个可变 Dataset 对象，事后修改其 transform 会同时影响二者。课程实现使用相同原始数据和互斥索引、不同 Dataset 实例来隔离 transform。

当 `num_workers>0` 时，DataLoader 会启动 worker 进程并预取样本。Dataset、`collate_fn` 和 worker 初始化逻辑需要可序列化；随机种子、内存复制、磁盘吞吐和进程启动成本都会影响性能与复现性。worker 越多不一定越快，应基于实际输入管线测量。

## 官方教程补充

**对应官方源文件：** `beginner_source/basics/data_tutorial.py`、`beginner_source/basics/transforms_tutorial.py`、`beginner_source/data_loading_tutorial.py`、`recipes_source/loading_data_recipe.rst`

官方教程把 `Dataset` 定义为索引到单样本的协议，把 `DataLoader` 定义为采样、组 batch 和并行加载的执行器。训练集可 `shuffle=True`，验证/测试集应保持稳定；transform 负责输入，target_transform 负责标签。增加 worker、pin memory 或异步拷贝前要测量瓶颈，并确保自定义 Dataset 在多进程环境中不会共享不安全状态。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

不运行代码，先回答再验证：

1. 实验 1 的两个 `Subset` 是否复制了 `features` 和 `labels`？它们真正保存了什么？
2. 为什么实验 2 有 3 个 batch，最后一个 batch 的形状是什么？设置 `drop_last=True` 后会怎样？
3. 为什么训练集通常需要 shuffle，而验证集通常不需要？对于按全量样本正确汇总的指标，验证顺序是否改变数学结果？
4. `model.eval()` 能否替代验证集的确定性 transform？为什么？
5. 实验 4 为什么为相同的 MNIST train split 创建两个 Dataset 实例，而不是先创建一个 Dataset 再切成两个 `Subset`？
6. quick 模式下，为什么验证集是 204 而不是对 1024 四舍五入得到 205？

## 试一试

1. **改变 batch 大小**：把实验 2 的 batch size 改为 7。先预测 batch 数和最后一批形状，再分别设置 `drop_last=False/True` 验证。
2. **固定采样顺序**：给 DataLoader 传入 `generator=torch.Generator().manual_seed(42)`，记录两个 epoch 的索引顺序；重新创建同种子的 loader，验证序列是否可复现。
3. **自定义组批**：为 `TensorDataset` 编写 `collate_fn`，返回字典 `{'inputs': ..., 'targets': ...}`，并在组批时把 inputs 标准化。思考哪些变换适合放在 Dataset，哪些适合放在 collate 阶段。
4. **检查增强隔离**：对实验 4 中同一个训练索引读取两次，观察随机增强可能产生不同张量；对同一个验证索引读取两次，使用 `torch.equal` 或 `torch.testing.assert_close` 验证其确定性。
5. **防止实体泄漏**：模拟每个用户拥有多个样本的数据，先做普通随机划分，再按 user id 分组划分，比较两种方式中用户是否跨集合出现。

## 常见错误与调试

- **划分后修改共享 Dataset 的 transform**：训练和验证可能一起改变。为不同数据流创建独立 Dataset 实例，并复用明确的索引集合。
- **先增强再划分**：同一原始样本的近重复版本可能跨集合出现，造成泄漏。先划分，再仅对训练路径在线增强。
- **对时间序列或用户数据逐样本随机划分**：未来信息或同一实体特征可能进入验证集。按时间、用户、设备或业务组划分。
- **验证集使用随机训练增强**：指标会波动且不再代表稳定的数据分布。验证和测试只保留必要的确定性预处理。
- **假设所有 batch 等大**：最后一批通常更小，导致 reshape 或指标分母错误。使用实际的 `inputs.size(0)`，或明确选择 `drop_last=True`。
- **错误平均 batch 指标**：大小不同的 batch 不能简单等权平均。按样本数加权累计损失和正确数。
- **`num_workers` 过大**：小数据集可能因进程启动和通信反而变慢，还可能放大内存使用。以 `0` 为基线逐步测量。
- **下载或路径失败**：实验 4 会把底层异常包装为 `RuntimeError`。先检查 `PYTORCH_ROOT / 'data'` 权限、缓存完整性与网络，再重试；不要把下载失败误判为模型问题。